# HR CTC Offer Analysis and Acceptance Recommender

Goal: use historical offer data to answer three HR questions:

1. What CTC range is suitable for a new candidate?
2. What are the 20th, 50th, and 80th percentile accepted CTC values for similar candidates?
3. What is the probability that a candidate accepts a given offered CTC?

Important: this is mainly a tabular machine learning and analytics problem. An LLM/agent can help explain results, inspect data quality, and guide users, but the salary and probability numbers should come from statistics/ML models trained on historical offer data.

## Dataset Columns

Current columns:

- `SLNO`
- `Candidate Ref`
- `Offer Date`
- `DOJ Extended`
- `Duration to accept offer`
- `Notice period`
- `Offered band`
- `Current CTC`
- `Expected CTC`
- `Offered CTC`
- `Negotiated CTC`
- `Final CTC`
- `Pecent hike expected in CTC`
- `Percent hike offered in CTC`
- `Percent difference CTC`
- `Joining Bonus`
- `Candidate relocate actual`
- `Gender`
- `Candidate Source`
- `Rex in Yrs`
- `LOB`
- `Primary Skill`
- `Previous Company Type`
- `Location`
- `Age`
- `Status`

## Most Important Fields

### Required for the MVP model

These are the most important fields for the acceptance prediction and CTC recommendation use case:

- `Status`: target variable. Convert `Joined` and `Accepted` to 1; convert `Declined` and `No Show` to 0.
- `Current CTC`: candidate's current annual compensation.
- `Expected CTC`: candidate's expected annual compensation.
- `Offered CTC`: offer amount made by the company. This is the key variable for acceptance probability.
- `Rex in Yrs`: relevant experience in years. Very important for CTC ranges.
- `Offered band`: job grade/level.
- `LOB`: line of business/domain.
- `Primary Skill`: main skill, for example Java, AWS, SAP, Testing.
- `Location`: salary expectations and talent markets vary by city.

### Useful additional features

These can improve prediction quality:

- `Notice period`: long notice periods can reduce joining probability.
- `Joining Bonus`: can increase acceptance probability.
- `Candidate relocate actual`: relocation can affect acceptance.
- `Candidate Source`: referrals, agencies, direct applicants may behave differently.
- `Previous Company Type`: product, service, startup, captive, consulting candidates can have different expectations.
- `Offer Date`: useful for market trend over time. Extract month, quarter, and year from it.

### Derived features

These are usually more useful than the raw CTC numbers alone:

- `expected_hike_pct = (Expected CTC - Current CTC) / Current CTC * 100`
- `offered_hike_pct = (Offered CTC - Current CTC) / Current CTC * 100`
- `offer_gap_pct = (Offered CTC - Expected CTC) / Expected CTC * 100`
- `offer_gap_amount = Offered CTC - Expected CTC`
- `experience_band`: bucketed version of `Rex in Yrs`.

The column `Pecent hike expected in CTC` has a typo. Rename it to `expected_hike_pct` or recalculate it from CTC fields.

## Fields to Avoid During Prediction

Some columns are useful for dashboards but risky for model training because they may be known only after the offer process has progressed. This is called target leakage.

Do not use these for predicting initial offer acceptance:

- `Duration to accept offer`: often known only after the candidate accepts.
- `DOJ Extended`: may happen after the offer decision.
- `Negotiated CTC`: known after negotiation, not at initial offer time.
- `Final CTC`: known after negotiation/finalization.

Use these for dashboards and separate analysis:

- offer-to-acceptance lag analysis
- negotiation analysis
- no-show analysis
- final CTC vs initial offered CTC comparison

## Fields to Exclude for Fairness

Do not use these as model inputs for salary recommendation or acceptance prediction:

- `Gender`
- `Age`

Keep them only for fairness auditing. For example, after training, check whether model errors or recommended CTC values are systematically worse for one group.

Important: simply removing gender and age is not enough. Other fields like `Location`, `Candidate Source`, or `Previous Company Type` can sometimes act as proxies. Monitor results by group.

## Recommended Model Strategy

Use two components:

### 1. Historical percentile recommender

For similar past candidates, calculate:

- 20th percentile accepted CTC
- 50th percentile accepted CTC
- 80th percentile accepted CTC
- acceptance rate
- count of similar records

This is highly explainable and HR-friendly.

### 2. Acceptance probability model

Train a classifier that predicts:

`P(candidate accepts | candidate profile, offered CTC)`

Candidate models:

- Baseline: logistic regression
- Good practical model: random forest
- Strong tabular models: XGBoost, LightGBM, CatBoost

For this dataset, CatBoost is a strong choice later because it handles categorical fields well. For the first MVP, use logistic regression or random forest with one-hot encoding.

## Reliability Checklist

To make the model reliable:

- Use a time-based train/test split when possible. Train on older offers and test on newer offers.
- Compare against simple baselines, such as average acceptance rate or a rule based on offer gap percentage.
- Track AUC-ROC, precision, recall, F1, and confusion matrix.
- Track calibration. If the model says 70% acceptance probability, roughly 70 out of 100 such offers should actually be accepted.
- Track Brier score for probability quality.
- Do not train on leakage columns.
- Keep a minimum similar-candidate count before showing percentile recommendations. For example, show a warning if fewer than 20 similar records are found.
- Re-train periodically as market salaries change.

The most important reliability metric for HR trust is calibration, not just accuracy.

## Fairness Checklist

To make the model fairer and safer:

- Exclude `Gender` and `Age` from model training.
- Use them only for auditing model behavior.
- Check acceptance prediction errors by gender and age bands.
- Check recommended CTC distribution by gender and age bands for similar candidate profiles.
- Check whether specific locations or sources are systematically under-recommended.
- Add product disclaimers: the model is decision support, not an automatic salary decision-maker.
- Always compare recommendation with compensation bands, internal parity, and HR policy.

A good product should show warnings like: `This recommendation is based on historical data and should be reviewed for internal parity and fairness before final offer.`

## Preprocessing Steps

Recommended preprocessing flow:

1. Load CSV/Excel.
2. Rename columns to clean snake_case names.
3. Convert CTC, percent, experience, notice period, and age fields to numeric.
4. Convert `Offer Date` to datetime and extract year/month/quarter.
5. Normalize Yes/No columns to 1/0.
6. Normalize categorical text fields by trimming spaces and standardizing casing.
7. Create `accepted` target from `Status`.
8. Create derived features: `expected_hike_pct`, `offered_hike_pct`, `offer_gap_pct`, `offer_gap_amount`.
9. Exclude identifier, leakage, and fairness-sensitive columns from model features.
10. Split train/test, preferably by offer date.
11. Train baseline model.
12. Evaluate model quality and calibration.
13. Build percentile recommender for similar candidates.

In [1]:
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(".matplotlib_cache").resolve()))

import pandas as pd
import numpy as np

DATA_PATH = "datasets/synthetic_hr_offer_acceptance_dataset.csv"

df = pd.read_csv(DATA_PATH)
df.head()

,SLNO,Candidate Ref,Offer Date,DOJ Extended,Duration to accept offer,Notice period,Offered band,Current CTC,Expected CTC,Offered CTC,...,Candidate relocate actual,Gender,Candidate Source,Rex in Yrs,LOB,Primary Skill,Previous Company Type,Location,Age,Status
0,1,2110001,2025-10-24,No,5,45,E3,15.70,24.83,19.30,...,No,Female,Agency,7.7,INFRA,Network,Captive,Mumbai,28,Declined
1,2,2110002,2025-04-04,No,2,45,E1,10.22,15.44,14.28,...,Yes,Male,LinkedIn,3.4,Digital,Node.js,Captive,Mumbai,31,Joined
2,3,2110003,2024-11-15,No,1,60,M1,31.97,62.14,48.08,...,No,Male,Agency,11.2,ERS,VLSI,Captive,Bangalore,35,Declined
3,4,2110004,2025-12-24,No,7,75,E1,6.32,9.20,9.03,...,Yes,Female,Direct,4.0,QA,API Testing,Consulting,Mumbai,28,Declined
4,5,2110005,2025-04-02,No,1,30,E2,11.94,16.50,18.37,...,No,Male,Employee Referral,8.7,ERS,Embedded C,Product,Chennai,36,No Show


In [2]:
column_map = {
    "SLNO": "slno",
    "Candidate Ref": "candidate_ref",
    "Offer Date": "offer_date",
    "DOJ Extended": "doj_extended",
    "Duration to accept offer": "duration_to_accept_days",
    "Notice period": "notice_period_days",
    "Offered band": "offered_band",
    "Current CTC": "current_ctc",
    "Expected CTC": "expected_ctc",
    "Offered CTC": "offered_ctc",
    "Negotiated CTC": "negotiated_ctc",
    "Final CTC": "final_ctc",
    "Pecent hike expected in CTC": "expected_hike_pct_original",
    "Percent hike offered in CTC": "offered_hike_pct_original",
    "Percent difference CTC": "offer_gap_pct_original",
    "Joining Bonus": "joining_bonus",
    "Candidate relocate actual": "relocation",
    "Gender": "gender",
    "Candidate Source": "candidate_source",
    "Rex in Yrs": "relevant_experience_years",
    "LOB": "lob",
    "Primary Skill": "primary_skill",
    "Previous Company Type": "previous_company_type",
    "Location": "location",
    "Age": "age",
    "Status": "status",
}

df = df.rename(columns=column_map)
df.columns

Index(['slno', 'candidate_ref', 'offer_date', 'doj_extended',
       'duration_to_accept_days', 'notice_period_days', 'offered_band',
       'current_ctc', 'expected_ctc', 'offered_ctc', 'negotiated_ctc',
       'final_ctc', 'expected_hike_pct_original', 'offered_hike_pct_original',
       'offer_gap_pct_original', 'joining_bonus', 'relocation', 'gender',
       'candidate_source', 'relevant_experience_years', 'lob', 'primary_skill',
       'previous_company_type', 'location', 'age', 'status'],
      dtype='str')

In [3]:
numeric_cols = [
    "duration_to_accept_days",
    "notice_period_days",
    "current_ctc",
    "expected_ctc",
    "offered_ctc",
    "negotiated_ctc",
    "final_ctc",
    "expected_hike_pct_original",
    "offered_hike_pct_original",
    "offer_gap_pct_original",
    "relevant_experience_years",
    "age",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["offer_date"] = pd.to_datetime(df["offer_date"], errors="coerce")
df["offer_year"] = df["offer_date"].dt.year
df["offer_month"] = df["offer_date"].dt.month
df["offer_quarter"] = df["offer_date"].dt.quarter

for col in ["joining_bonus", "relocation", "doj_extended"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})

categorical_cols = ["offered_band", "candidate_source", "lob", "primary_skill", "previous_company_type", "location", "status"]
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

tier_1_cities = {"Bangalore", "Mumbai", "Delhi", "NCR", "Hyderabad", "Chennai", "Pune", "Kolkata"}
tier_2_cities = {"Noida", "Gurgaon", "Gurugram", "Ahmedabad", "Kochi", "Coimbatore", "Indore", "Jaipur", "Chandigarh", "Mysore"}

def city_tier(location):
    if pd.isna(location):
        return "Unknown"
    location = str(location).strip()
    if location in tier_1_cities:
        return "Tier 1"
    if location in tier_2_cities:
        return "Tier 2"
    return "Tier 3/Other"

df["city_tier"] = df["location"].apply(city_tier)

accepted_statuses = {"Joined", "Accepted"}
df["accepted"] = df["status"].isin(accepted_statuses).astype(int)

df["expected_hike_pct"] = (df["expected_ctc"] - df["current_ctc"]) / df["current_ctc"] * 100
df["offered_hike_pct"] = (df["offered_ctc"] - df["current_ctc"]) / df["current_ctc"] * 100
df["offer_gap_pct"] = (df["offered_ctc"] - df["expected_ctc"]) / df["expected_ctc"] * 100
df["offer_gap_amount"] = df["offered_ctc"] - df["expected_ctc"]

df[["status", "accepted", "current_ctc", "expected_ctc", "offered_ctc", "offer_gap_pct"]].head()

,status,accepted,current_ctc,expected_ctc,offered_ctc,offer_gap_pct
0,Declined,0,15.70,24.83,19.30,-22.271446
1,Joined,1,10.22,15.44,14.28,-7.512953
2,Declined,0,31.97,62.14,48.08,-22.626328
3,Declined,0,6.32,9.20,9.03,-1.847826
4,No Show,0,11.94,16.50,18.37,11.333333


## Feature Sets

`target_col` is what the model predicts.

`model_features` are safe features known before or at offer time.

`excluded_cols` are excluded because they are identifiers, leakage risks, sensitive fields, or raw columns replaced by derived features.

In [5]:
target_col = "accepted"

model_features = [
    "current_ctc", 
    "expected_ctc",
    "offered_ctc",
    "relevant_experience_years",
    "notice_period_days",
    "joining_bonus",
    "relocation",
    "offer_year",
    "offer_month",
    "offer_quarter",
    "expected_hike_pct",
    "offered_hike_pct",
    "offer_gap_pct",
    "offer_gap_amount",
    "offered_band",
    "candidate_source",
    "lob",
    "primary_skill",
    "previous_company_type",
    "location",
    "city_tier",
]

excluded_cols = [
    "slno",
    "candidate_ref",
    "status",
    "offer_date",
    "duration_to_accept_days",
    "doj_extended",
    "negotiated_ctc",
    "final_ctc",
    "gender",
    "age",
    "expected_hike_pct_original",
    "offered_hike_pct_original",
    "offer_gap_pct_original",
]

X = df[model_features]
y = df[target_col]

X.head()

,current_ctc,expected_ctc,offered_ctc,relevant_experience_years,notice_period_days,joining_bonus,relocation,offer_year,offer_month,offer_quarter,...,offered_hike_pct,offer_gap_pct,offer_gap_amount,offered_band,candidate_source,lob,primary_skill,previous_company_type,location,city_tier
0,15.70,24.83,19.30,7.7,45,0,0,2025,10,4,...,22.929936,-22.271446,-5.53,E3,Agency,INFRA,Network,Captive,Mumbai,Tier 1
1,10.22,15.44,14.28,3.4,45,1,1,2025,4,2,...,39.726027,-7.512953,-1.16,E1,LinkedIn,Digital,Node.js,Captive,Mumbai,Tier 1
2,31.97,62.14,48.08,11.2,60,1,0,2024,11,4,...,50.390992,-22.626328,-14.06,M1,Agency,ERS,VLSI,Captive,Bangalore,Tier 1
3,6.32,9.20,9.03,4.0,75,1,1,2025,12,4,...,42.879747,-1.847826,-0.17,E1,Direct,QA,API Testing,Consulting,Mumbai,Tier 1
4,11.94,16.50,18.37,8.7,30,1,0,2025,4,2,...,53.852596,11.333333,1.87,E2,Employee Referral,ERS,Embedded C,Product,Chennai,Tier 1


## Baseline Acceptance Model

Start with logistic regression. It is simple, explainable, and a good baseline.

After that, try random forest. Later, try CatBoost/XGBoost/LightGBM if available.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, brier_score_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
print("ROC AUC:", round(roc_auc_score(y_test, proba), 3))
print("Brier score:", round(brier_score_loss(y_test, proba), 3))

              precision    recall  f1-score   support

           0       0.70      0.76      0.73        78
           1       0.75      0.70      0.72        82

    accuracy                           0.72       160
   macro avg       0.73      0.73      0.72       160
weighted avg       0.73      0.72      0.72       160

ROC AUC: 0.764
Brier score: 0.197


## Compare Multiple Models

Yes, this is where you try different models. Keep the same `preprocess` step and swap only the classifier.

Use the same train/test split for every model. Then compare:

- `roc_auc`: ranking quality. Higher is better.
- `brier_score`: probability quality. Lower is better.
- `f1`: balance between precision and recall. Higher is better.
- `precision`: when the model says accept, how often is it right?
- `recall`: how many actual acceptances does it catch?

For this HR use case, do not choose only by accuracy. Prefer a model with good ROC AUC and good Brier score because HR needs reliable probabilities, not just class labels.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier

candidate_models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        random_state=42,
    ),
    "xgboost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        scale_pos_weight=1,
        random_state=42,
    )
    
}

model_results = []
trained_models = {}

for name, classifier in candidate_models.items():
    candidate_pipeline = Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("classifier", classifier),
        ]
    )

    candidate_pipeline.fit(X_train, y_train)
    pred = candidate_pipeline.predict(X_test)
    proba = candidate_pipeline.predict_proba(X_test)[:, 1]

    model_results.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(y_test, pred),
            "recall": recall_score(y_test, pred),
            "f1": f1_score(y_test, pred),
            "roc_auc": roc_auc_score(y_test, proba),
            "brier_score": brier_score_loss(y_test, proba),
        }
    )
    trained_models[name] = candidate_pipeline

model_comparison = pd.DataFrame(model_results).sort_values(
    by=["roc_auc", "brier_score"],
    ascending=[False, True],
)

model_comparison

,model,accuracy,precision,recall,f1,roc_auc,brier_score
0,logistic_regression,0.72500,0.750000,0.695122,0.721519,0.763915,0.197136
1,random_forest,0.68750,0.728571,0.621951,0.671053,0.739212,0.206670
2,hist_gradient_boosting,0.66250,0.684211,0.634146,0.658228,0.704034,0.241718
3,xgboost,0.64375,0.647059,0.670732,0.658683,0.699969,0.248164


After comparing models, choose a final model deliberately.

A good first rule:

- If logistic regression is close to the best model, keep logistic regression because it is simpler and easier to explain.
- If random forest or gradient boosting is clearly better on ROC AUC and Brier score, use that model.
- If Brier score is poor, add calibration before showing probabilities to HR.

Example:

`model = trained_models["logistic_regression"]`

or

`model = trained_models["random_forest"]`

## Historical Percentile Recommender

For CTC suggestion, do not rely only on the classifier. Also calculate accepted CTC percentiles from similar historical candidates.

This version filters by skill, LOB, location, and band. In a real product, make this fallback gracefully if too few records are found.

In [13]:
def filter_by_fields(data, filters):
    subset = data.copy()
    applied_filters = {}

    for col, value in filters.items():
        if value is None or col not in subset.columns:
            continue
        subset = subset[subset[col] == value]
        applied_filters[col] = value

    return subset, applied_filters


def summarize_ctc_segment(subset, applied_filters, min_records):
    accepted_subset = subset[subset["accepted"] == 1]
    percentiles = accepted_subset["offered_ctc"].quantile([0.2, 0.5, 0.8]).to_dict()

    warning = None
    if len(accepted_subset) < min_records:
        warning = (
            f"Only {len(accepted_subset)} accepted records found. "
            "Broaden filters or treat recommendation cautiously."
        )

    return {
        "filters_used": applied_filters,
        "similar_records": len(subset),
        "accepted_similar_records": len(accepted_subset),
        "acceptance_rate": subset["accepted"].mean() if len(subset) else np.nan,
        "p20_offered_ctc": percentiles.get(0.2, np.nan),
        "p50_offered_ctc": percentiles.get(0.5, np.nan),
        "p80_offered_ctc": percentiles.get(0.8, np.nan),
        "warning": warning,
    }


def flexible_accepted_ctc_percentiles(
    data,
    primary_skill=None,
    lob=None,
    location=None,
    city_tier=None,
    offered_band=None,
    min_records=20,
    flexibility="balanced",
):
    """
    flexibility options:
    - strict: tries highly specific filters first.
    - balanced: starts specific, then quickly falls back to broader business segments.
    - broad: avoids over-filtering and prefers larger comparison groups.
    """

    exact_filters = {
        "primary_skill": primary_skill,
        "lob": lob,
        "location": location,
        "city_tier": city_tier,
        "offered_band": offered_band,
    }

    fallback_levels = {
        "strict": [
            ["primary_skill", "lob", "location", "offered_band"],
            ["primary_skill", "lob", "city_tier", "offered_band"],
            ["primary_skill", "lob", "offered_band"],
            ["lob", "location", "offered_band"],
            ["lob", "city_tier", "offered_band"],
            ["lob", "offered_band"],
            ["offered_band"],
        ],
        "balanced": [
            ["primary_skill", "lob", "location", "offered_band"],
            ["primary_skill", "lob", "offered_band"],
            ["lob", "city_tier", "offered_band"],
            ["lob", "offered_band"],
            ["primary_skill", "lob"],
            ["offered_band"],
        ],
        "broad": [
            ["lob", "city_tier", "offered_band"],
            ["lob", "offered_band"],
            ["primary_skill", "lob"],
            ["lob"],
            ["offered_band"],
            [],
        ],
    }

    if flexibility not in fallback_levels:
        raise ValueError("flexibility must be one of: strict, balanced, broad")

    attempts = []
    best_result = None

    for level in fallback_levels[flexibility]:
        filters = {field: exact_filters[field] for field in level if exact_filters.get(field) is not None}
        subset, applied_filters = filter_by_fields(data, filters)
        result = summarize_ctc_segment(subset, applied_filters, min_records)
        attempts.append(result)

        if result["accepted_similar_records"] >= min_records:
            result["fallback_attempts"] = attempts
            result["confidence"] = "High" if result["accepted_similar_records"] >= min_records * 2 else "Medium"
            return result

        if best_result is None or result["accepted_similar_records"] > best_result["accepted_similar_records"]:
            best_result = result

    best_result["fallback_attempts"] = attempts
    best_result["confidence"] = "Low"
    return best_result


flexible_accepted_ctc_percentiles(
    df,
    primary_skill="Network",
    lob="INFRA",
    location="Mumbai",
    city_tier="Tier 1",
    offered_band="E3",
    flexibility="balanced",
)

{'filters_used': {'offered_band': 'E3'},
 'similar_records': 218,
 'accepted_similar_records': 106,
 'acceptance_rate': np.float64(0.48623853211009177),
 'p20_offered_ctc': 18.68,
 'p50_offered_ctc': 24.92,
 'p80_offered_ctc': 32.51,
 'warning': None,
 'fallback_attempts': [{'filters_used': {'primary_skill': 'Network',
    'lob': 'INFRA',
    'location': 'Mumbai',
    'offered_band': 'E3'},
   'similar_records': 3,
   'accepted_similar_records': 0,
   'acceptance_rate': np.float64(0.0),
   'p20_offered_ctc': nan,
   'p50_offered_ctc': nan,
   'p80_offered_ctc': nan,
   'warning': 'Only 0 accepted records found. Broaden filters or treat recommendation cautiously.'},
  {'filters_used': {'primary_skill': 'Network',
    'lob': 'INFRA',
    'offered_band': 'E3'},
   'similar_records': 6,
   'accepted_similar_records': 1,
   'acceptance_rate': np.float64(0.16666666666666666),
   'p20_offered_ctc': 17.85,
   'p50_offered_ctc': 17.85,
   'p80_offered_ctc': 17.85,
   'warning': 'Only 1 accepted

## Predict Acceptance at Different CTC Offers

For a candidate, vary `offered_ctc` across a range and calculate predicted acceptance probability. This creates the acceptance-probability curve that HR can use.

In [ ]:
def acceptance_curve(candidate_row, offer_values, trained_model):
    rows = []
    for offer in offer_values:
        row = candidate_row.copy()
        row["offered_ctc"] = offer
        row["offered_hike_pct"] = (row["offered_ctc"] - row["current_ctc"]) / row["current_ctc"] * 100
        row["offer_gap_pct"] = (row["offered_ctc"] - row["expected_ctc"]) / row["expected_ctc"] * 100
        row["offer_gap_amount"] = row["offered_ctc"] - row["expected_ctc"]
        rows.append(row)

    curve_df = pd.DataFrame(rows)[model_features]
    curve_df["acceptance_probability"] = trained_model.predict_proba(curve_df)[:, 1]
    return curve_df[["offered_ctc", "acceptance_probability"]]

def acceptance_curve(candidate_row, offer_values, trained_model):
    rows = []
    for offer in offer_values:
        row = candidate_row.copy()
        row["offered_ctc"] = offer
        row["offered_hike_pct"] = (row["offered_ctc"] - row["current_ctc"]) / row["current_ctc"] * 100
        row["offer_gap_pct"] = (row["offered_ctc"] - row["expected_ctc"]) / row["expected_ctc"] * 100
        row["offer_gap_amount"] = row["offered_ctc"] - row["expected_ctc"]
        rows.append(row)

    curve_df = pd.DataFrame(rows)[model_features]
    curve_df["acceptance_probability"] = trained_model.predict_proba(curve_df)[:, 1]
    return curve_df[["offered_ctc", "acceptance_probability"]]


# Sample 5 random candidates from the held-out test set (not part of training)
print("Sampling 5 random candidates from X_test (held-out, not in training).")
candidates = X_test.sample(5, random_state=42)

for idx, row in candidates.iterrows():
    cand = row.copy()
    offer_values = np.linspace(cand["current_ctc"] * 1.05, cand["expected_ctc"] * 1.15, 20)
    curve = acceptance_curve(cand, offer_values, model)
    print(f"\nCandidate index {idx}: location={cand['location']}, skill={cand['primary_skill']}, band={cand['offered_band']}")
    print(curve.to_string(index=False))

current_ctc                      31.97
expected_ctc                     62.14
offered_ctc                      48.08
relevant_experience_years         11.2
notice_period_days                  60
joining_bonus                      NaN
relocation                         NaN
offer_year                        2024
offer_month                         11
offer_quarter                        4
expected_hike_pct            94.369722
offered_hike_pct             50.390992
offer_gap_pct               -22.626328
offer_gap_amount                -14.06
offered_band                        M1
candidate_source                Agency
lob                                ERS
primary_skill                     VLSI
previous_company_type          Captive
location                     Bangalore
city_tier                       Tier 1
Name: 2, dtype: object


c:\Users\Dell\Documents\Codex\2026-07-05\what-about-this-based-on-the\ctc_recommender\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['joining_bonus' 'relocation']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


,offered_ctc,acceptance_probability
2,33.568500,0.027439
2,35.562842,0.037163
2,37.557184,0.050157
2,39.551526,0.067374
2,41.545868,0.089943
2,43.540211,0.119107
2,45.534553,0.156104
2,47.528895,0.201959
2,49.523237,0.257179
2,51.517579,0.321416


## Next Improvements

1. Add a time-based split using `Offer Date` instead of random split.
2. Add calibration plots and probability buckets.
3. Add fairness audits using `Gender` and `Age` only after prediction.
4. Try random forest and CatBoost/LightGBM/XGBoost.
5. Build dashboard views: offer funnel, accepted vs declined CTC distributions, CTC percentiles by segment, and acceptance probability curve.
6. Build a schema mapper so real customer Excel files can have different column names.